# Notebook 02 — Behavioral baselines and random-row ICL null (Gate 1)

Validate the evaluator; reproduce that random eagle-number rows do NOT create a monotonic diagonal. Full run needs a model; fast path checks the scoring math.

In [ ]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


## 1. Configuration and run manifest

In [ ]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="02", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


## 2. Preflight assertions

In [ ]:
print('preflight: package + numpy import OK')

## 3. Load or compute cached artifacts
Full-sequence logprob matches a manual token loop (fixture); target margin behaves.

In [ ]:
from subliminal_icl.token_scoring import token_logprobs, token_logprobs_manual, target_margin, top1_rate
rng = np.random.default_rng(0); logits = rng.standard_normal((4, 13)); ids = [2, 5, 1, 7]
match = np.allclose(token_logprobs(logits, ids), token_logprobs_manual(logits, ids))
lp = {"eagle": -1.0, "cat": -2.0, "dog": -2.2, "owl": -3.0}
fT = target_margin(lp, "eagle"); print("F_eagle:", round(fT, 3), "top1:", top1_rate(lp, "eagle"))
print("full-seq==manual:", match)

## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
See printed tables above; plotting helpers in `subliminal_icl.plotting`.

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [ ]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("fullseq_matches_manual", match, "no first-token shortcut"),
          ("target_margin_positive_control", fT > 0, "explicit target margin positive")]
gs = run_gate_checks("gate_01_evaluator", "Evaluator sensitivity/specificity", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


## 7. Write checkpoint report
Written by the gate cell when not in FAST_DEV_RUN.